# Yoruba OCR - Lightning AI GPU runner

Use this notebook inside a Lightning AI Studio with a GPU machine. It assumes the repo lives in the Studio filesystem, typically under `/teamspace/studios/this_studio/yoruba_ocr_research`, and data is either already inside `data/processed` or available at `DATA_ROOT`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

STUDIO = Path('/teamspace/studios/this_studio') if Path('/teamspace/studios/this_studio').exists() else Path.cwd()
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', STUDIO / 'yoruba_ocr_research'))
GITHUB_REPO = os.environ.get('GITHUB_REPO', '')

if not PROJECT_ROOT.exists():
    if GITHUB_REPO:
        subprocess.check_call(['git', 'clone', GITHUB_REPO, str(PROJECT_ROOT)])
    elif Path.cwd().name == 'yoruba_ocr_research':
        PROJECT_ROOT = Path.cwd()
    else:
        raise RuntimeError('Set PROJECT_ROOT to the repo path or set GITHUB_REPO to clone it.')

os.chdir(PROJECT_ROOT)
os.environ['PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['PYTHON'] = sys.executable
os.environ['HF_HOME'] = str(PROJECT_ROOT / '.hf_cache')
os.environ['HF_HUB_CACHE'] = os.environ['HF_HOME']
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.executable)

## Attach or link data

If `data/processed` is missing, set `DATA_ROOT` to either a folder containing `processed/` or the `processed/` folder itself.

In [ ]:
processed = PROJECT_ROOT / 'data' / 'processed'
if not processed.exists():
    data_root = Path(os.environ.get('DATA_ROOT', STUDIO / 'data'))
    candidates = [data_root / 'processed', data_root, STUDIO / 'data' / 'processed']
    source = next((p for p in candidates if (p / 'labels' / 'test.txt').exists()), None)
    if source is None:
        raise RuntimeError('Could not find processed data. Upload it and set DATA_ROOT.')
    (PROJECT_ROOT / 'data').mkdir(exist_ok=True)
    processed.symlink_to(source, target_is_directory=True)
    print('Linked data/processed ->', source)
else:
    print('Found existing data/processed:', processed)

for rel in ['labels/train.txt', 'labels/val.txt', 'labels/test.txt', 'dictionary/yoruba_char_dict.txt']:
    path = processed / rel
    if not path.exists():
        raise FileNotFoundError(path)
print('Data layout OK')

## Install dependencies

Use the Studio GPU image. If Paddle GPU install fails, set `PADDLE_PIP_SPEC` to the exact wheel for the CUDA version shown by `nvidia-smi`.

In [ ]:
def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

subprocess.call(['nvidia-smi'])
pip_install('-U', 'pip')
pip_install(os.environ.get('PADDLE_PIP_SPEC', 'paddlepaddle-gpu>=2.6,<2.7'))
pip_install('-r', 'requirements.txt')

if not (PROJECT_ROOT / 'PaddleOCR').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/PaddlePaddle/PaddleOCR.git', 'PaddleOCR'])
pip_install('-r', 'PaddleOCR/requirements.txt')
pip_install('-U', 'transformers>=5.0.0', 'accelerate>=1.1.0', 'huggingface_hub>=1.5.0', 'datasets', 'safetensors', 'einops', 'torchvision', 'bitsandbytes')

import paddle, torch, transformers, accelerate
print('Paddle CUDA devices:', paddle.device.cuda.device_count())
print('Torch CUDA:', torch.cuda.is_available())
print('Transformers:', transformers.__version__)
if int(transformers.__version__.split('.')[0]) < 5:
    raise RuntimeError('Restart the Studio kernel, then rerun setup. transformers>=5 is required.')

## Run plan

Lightning has persistent disk, so SFT is more practical here than on shorter notebook sessions.

In [ ]:
RUN_RESET = True
RUN_PADDLE_BASELINE = True
RUN_VLM_ZERO_SHOT = True
RUN_PADDLEOCRVL16_SFT = False
RUN_ANALYSIS = True

os.environ['EVAL_USE_GPU'] = '1'
os.environ['CONFIG_FORCE_GPU'] = '1'
os.environ['PADDLEOCRVL16_QUANTIZE_4BIT'] = os.environ.get('PADDLEOCRVL16_QUANTIZE_4BIT', '0')
os.environ['GLM_QUANTIZE_4BIT'] = os.environ.get('GLM_QUANTIZE_4BIT', '0')

def run(cmd):
    print('$', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=PROJECT_ROOT, env=os.environ.copy())

In [ ]:
if RUN_RESET:
    run([sys.executable, 'scripts/metrics_lifecycle.py', 'reset'])

run(['bash', 'scripts/shell/phase_02_analyze.sh'])
run(['bash', 'scripts/shell/phase_03_config.sh'])

if RUN_PADDLE_BASELINE:
    run(['bash', 'scripts/shell/phase_05_eval_paddleocr_recognition.sh'])

if RUN_VLM_ZERO_SHOT:
    run(['bash', 'scripts/shell/phase_15_eval_paddleocrvl16_zero_shot.sh'])
    run(['bash', 'scripts/shell/phase_18_eval_glm_ocr_zero_shot.sh'])

if RUN_PADDLEOCRVL16_SFT:
    run(['bash', 'scripts/shell/phase_14_export_paddleocrvl16_sft.sh'])
    run(['bash', 'scripts/shell/phase_16_train_paddleocrvl16_sft.sh'])
    run(['bash', 'scripts/shell/phase_17_eval_paddleocrvl16_sft.sh'])

if RUN_ANALYSIS:
    for script in ['17_stratified_error_analysis.py', '18_der_universe_ablation.py', '19_bootstrap_metric_cis.py']:
        run([sys.executable, f'scripts/{script}'])
    run([sys.executable, 'scripts/11_compile_results.py'])
    run([sys.executable, 'scripts/22_generate_plots.py'])
    run([sys.executable, 'scripts/23_write_research_approach.py', '--output', 'research_approach.md'])

## Package outputs

Outputs remain in the Studio filesystem. This cell also creates a zip for download.

In [ ]:
export_root = PROJECT_ROOT / 'platform_outputs' / 'lightning_ai'
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True)
for name in ['results', 'research_approach.md']:
    src = PROJECT_ROOT / name
    if src.is_dir():
        shutil.copytree(src, export_root / name)
    elif src.is_file():
        shutil.copy2(src, export_root / name)
if (PROJECT_ROOT / 'experiments').exists() and RUN_PADDLEOCRVL16_SFT:
    shutil.copytree(PROJECT_ROOT / 'experiments', export_root / 'experiments')
archive = shutil.make_archive(str(export_root), 'zip', export_root)
print('Exported:', archive)